# Считайте датасет из файла

In [23]:
import pandas as pd
import numpy as np
df = pd.read_csv("train.csv")

# Выведите основную информацию о датасете: информацию о типах данных, число пропусков, средние значения и т.д

In [24]:
df_info = pd.DataFrame({
    'Тип данных': df.dtypes,
    'Пропуски': df.isnull().sum(),
    'Уникальные': df.nunique()
})
print(df_info)
print("\nСредние значения:")
print(df.mean(numeric_only=True))

            Тип данных  Пропуски  Уникальные
PassengerId      int64         0         891
Survived         int64         0           2
Pclass           int64         0           3
Name            object         0         891
Sex             object         0           2
Age            float64       177          88
SibSp            int64         0           7
Parch            int64         0           7
Ticket          object         0         681
Fare           float64         0         248
Cabin           object       687         147
Embarked        object         2           3

Средние значения:
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64


#Посчитайте процент выживаемости у каждого класса пассажиров (Pclass)

In [25]:
survival_by_class = df.groupby('Pclass')['Survived'].mean() * 100
survival_by_class = survival_by_class.round(2)
print(f"\nПроцент выживаемости по классам:\n{survival_by_class}")


Процент выживаемости по классам:
Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64


#Выведите самое популярное мужское и самое популярное женское имя на корабле

In [26]:
def extract_name(name, sex):
    if sex == 'female' and '(' in name and ')' in name:
        bracket_match = re.search(r'\((.*?)\)', name)
        if bracket_match:
            return bracket_match.group(1).strip().split()[0]
    match = re.search(r',\s+[A-Za-z]+\.\s+([A-Za-z]+)', name)
    if match:
        return match.group(1)
    return "Unknown"

df['First_Name'] = df.apply(lambda row: extract_name(row['Name'], row['Sex']), axis=1)

In [27]:
male_names = df[df['Sex'] == 'male']['First_Name'].value_counts()
female_names = df[df['Sex'] == 'female']['First_Name'].value_counts()

print(f"\nСамое популярное мужское имя: {male_names.index[0]} ({male_names.iloc[0]} раз)")
print(f"Самое популярное женское имя: {female_names.index[0]} ({female_names.iloc[0]} раз)")


Самое популярное мужское имя: William (35 раз)
Самое популярное женское имя: Anna (15 раз)


#Выведите самое популярное мужское и самое популярное женское имя на корабле в каждом класс

In [28]:
print("\nСамое популярное имя в каждом классе:")
for pclass in sorted(df['Pclass'].unique()):
    class_data = df[df['Pclass'] == pclass]
    male_class = class_data[class_data['Sex'] == 'male']['First_Name'].value_counts()
    female_class = class_data[class_data['Sex'] == 'female']['First_Name'].value_counts()
    male_name = f"{male_class.index[0]} ({male_class.iloc[0]})" if not male_class.empty else "нет"
    female_name = f"{female_class.index[0]} ({female_class.iloc[0]})" if not female_class.empty else "нет"
    print(f"  Класс {pclass}: мужские - {male_name}, женские - {female_name}")


Самое популярное имя в каждом классе:
  Класс 1: мужские - William (11), женские - Elizabeth (5)
  Класс 2: мужские - William (9), женские - Elizabeth (5)
  Класс 3: мужские - William (15), женские - Anna (9)


#Выведите часть таблицы с пассажирами, возраст которых больше 44 лет

In [29]:
older_44 = df[df['Age'] > 44]
print(f"\nПассажиры старше 44 лет: {len(older_44)} человек")
print(older_44[['Name', 'Age', 'Sex', 'Pclass']].head())


Пассажиры старше 44 лет: 115 человек
                                        Name   Age     Sex  Pclass
6                    McCarthy, Mr. Timothy J  54.0    male       1
11                  Bonnell, Miss. Elizabeth  58.0  female       1
15          Hewlett, Mrs. (Mary D Kingcome)   55.0  female       2
33                     Wheadon, Mr. Edward H  66.0    male       2
52  Harper, Mrs. Henry Sleeper (Myna Haxtun)  49.0  female       1


#Выведите часть таблицы с пассажирами, возраст которых меньше 44 лет и которые мужского пол

In [30]:
male_under_44 = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
print(f"\nМужчины младше 44 лет: {len(male_under_44)} человек")
print(male_under_44[['Name', 'Age', 'Sex', 'Pclass']].head())


Мужчины младше 44 лет: 368 человек
                              Name   Age   Sex  Pclass
0          Braund, Mr. Owen Harris  22.0  male       3
4         Allen, Mr. William Henry  35.0  male       3
7   Palsson, Master. Gosta Leonard   2.0  male       3
12  Saundercock, Mr. William Henry  20.0  male       3
13     Andersson, Mr. Anders Johan  39.0  male       3


#Выведите количества n-местных кабин (в которых было 2, 3, 4, ... человека)

In [31]:
cabins_df = df[['PassengerId', 'Cabin']].copy()
cabins_df = cabins_df[cabins_df['Cabin'].notna()]
cabins_df['Cabin_List'] = cabins_df['Cabin'].str.split()
cabin_passengers = []
for idx, row in cabins_df.iterrows():
    for cabin in row['Cabin_List']:
        cabin_passengers.append({'Cabin': cabin, 'PassengerId': row['PassengerId']})

cabin_distribution = pd.DataFrame(cabin_passengers)

cabin_sizes = cabin_distribution.groupby('Cabin').size().reset_index(name='passenger_count')

n_place_cabins = cabin_sizes['passenger_count'].value_counts().sort_index()

print("\nКоличество n-местных кабин:")
for places, count in n_place_cabins.items():
    print(f"  {places} место(а): {count} кают(ы)")


Количество n-местных кабин:
  1 место(а): 104 кают(ы)
  2 место(а): 44 кают(ы)
  3 место(а): 6 кают(ы)
  4 место(а): 7 кают(ы)


#Выведите количество пассажиров, у которых нет родственников на борту

In [32]:
no_relatives = df[(df['SibSp'] == 0) & (df['Parch'] == 0)]
print(f"\nПассажиров без родственников на борту: {len(no_relatives)}")


Пассажиров без родственников на борту: 537
